In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Hierarchical Triage Meta-Ensemble (`models/meta_ensemble_esi1_esi2345.ipynb`)

This notebook builds a **5-Class Hierarchical Meta-Ensemble** combining two trained model artifacts:
1. **Stage 1 Model**: `deploy/xgboost_raw_esi1_extreme_model.rds` (Binary XGBoost predicting $P(\text{ESI } 1)$ vs $P(\text{Not ESI } 1)$).
2. **Stage 2 Model**: `deploy/lightbgm_feng_esi2345_extreme_model.rds` (4-Class LightGBM predicting $P(\text{ESI } k \mid \text{Not ESI } 1)$ for $k \in \{2, 3, 4, 5\}$).

### Hierarchical Probability Law
$$\mathbf{P}(\text{ESI} = 1) = P_1$$
$$\mathbf{P}(\text{ESI} = k) = (1 - P_1) \cdot Q_k \quad \text{for } k \in \{2, 3, 4, 5\}$$

### Evaluated Metrics
- **ROC-AUC** (Macro One-vs-Rest ROC-AUC and per-class ROC-AUC)
- **Accuracy** (Overall 5-class classification accuracy)
- **Precision** (Macro Precision & per-class precision)
- **Recall** (Macro Recall / Sensitivity & per-class recall)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Model Artifacts & Construct Full 19 Predictor Dataset
# ---------------------------------------------------------
set.seed(config$training$random_state)
model1_path <- "../deploy/xgboost_raw_esi1_extreme_model.rds"
if (!file.exists(model1_path)) model1_path <- "deploy/xgboost_raw_esi1_extreme_model.rds"
model2_path <- "../deploy/lightbgm_feng_esi2345_extreme_model.rds"
if (!file.exists(model2_path)) model2_path <- "deploy/lightbgm_feng_esi2345_extreme_model.rds"
cat("Loading Stage 1 Model (Binary ESI 1 XGBoost):", model1_path, "...\n")
model1_obj <- readRDS(model1_path)
cat("Loading Stage 2 Model (4-Class ESI 2345 LightGBM):", model2_path, "...\n")
model2_obj <- readRDS(model2_path)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("\nLoading main evaluation dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 19 Predictors
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col]])
df_full$target_layer1 <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA raw features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Evaluation Dataset Ready: %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Natural 5-Class Target Distribution ('1', '2', '3', '4', '5'):\n")
print(table(df_full$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Hierarchical Inference Pipeline
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df_full$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# HIERARCHICAL ENSEMBLE INFERENCE FUNCTION
predict_hierarchical_ensemble <- function(df_split, m1_obj, m2_obj) {
  feat_names <- c("age", "gender", "cc_breathingdifficulty",
                  "hr_mean_to_last", "sbp_mean_to_last", "spo2_mean_to_last", "rr_mean_to_last",
                  "hr_range", "rr_range", "spo2_range", "sbp_range",
                  "hr_last_to_min", "rr_last_to_min", "spo2_last_to_min", "sbp_last_to_min",
                  "hr_last_to_max", "rr_last_to_max", "spo2_last_to_max", "sbp_last_to_max")
  
  # Standardize using trained preprocessors
  df_m1 <- predict(m1_obj$preproc, df_split)
  df_m2 <- predict(m2_obj$preproc, df_split)
  
  X_m1 <- as.matrix(df_m1[, feat_names])
  X_m2 <- as.matrix(df_m2[, feat_names])
  
  # Stage 1: XGBoost Binary ESI 1
  dmatrix1 <- xgb.DMatrix(data = X_m1)
  p1 <- predict(m1_obj$model, dmatrix1) # P(ESI 1)
  
  # Stage 2: LightGBM 4-Class (ESI 2, 3, 4, 5)
  raw_preds2 <- predict(m2_obj$model, X_m2)
  if (is.matrix(raw_preds2)) {
    q_mat <- raw_preds2
  } else {
    q_mat <- matrix(raw_preds2, ncol = 4, byrow = TRUE)
  }
  colnames(q_mat) <- c("2", "3", "4", "5")
  
  # Hierarchical Probability Product: P(ESI k) = (1 - p1) * q_k for k in 2..5
  prob_5class <- matrix(0, nrow = nrow(df_split), ncol = 5)
  colnames(prob_5class) <- c("1", "2", "3", "4", "5")
  
  prob_5class[, "1"] <- p1
  prob_5class[, "2"] <- (1 - p1) * q_mat[, "2"]
  prob_5class[, "3"] <- (1 - p1) * q_mat[, "3"]
  prob_5class[, "4"] <- (1 - p1) * q_mat[, "4"]
  prob_5class[, "5"] <- (1 - p1) * q_mat[, "5"]
  
  return(prob_5class)
}
cat("Generating Hierarchical Ensemble 5-Class Probabilities across splits...\n")
prob_train <- predict_hierarchical_ensemble(train_df, model1_obj, model2_obj)
prob_val   <- predict_hierarchical_ensemble(val_df,   model1_obj, model2_obj)
prob_test  <- predict_hierarchical_ensemble(test_df,  model1_obj, model2_obj)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Comprehensive 5-Class Evaluation (Accuracy, Precision, Recall, ROC-AUC)
# ---------------------------------------------------------
evaluate_ensemble_5class <- function(prob_mat, actual_factor, set_name) {
  pred_indices <- apply(prob_mat, 1, which.max)
  pred_val     <- colnames(prob_mat)[pred_indices]
  pred_fac     <- factor(pred_val, levels = c("1", "2", "3", "4", "5"))
  act_fac      <- factor(actual_factor, levels = c("1", "2", "3", "4", "5"))
  
  cm  <- confusionMatrix(pred_fac, act_fac)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_class <- as.numeric(cm$byClass[, "Pos Pred Value"])
  rec_by_class  <- as.numeric(cm$byClass[, "Sensitivity"])
  prec_by_class[is.na(prec_by_class)] <- 0
  rec_by_class[is.na(rec_by_class)]   <- 0
  
  # One-vs-Rest ROC-AUC for each of the 5 classes
  roc_auc_by_class <- sapply(1:5, function(i) {
    cls_name <- levels(act_fac)[i]
    act_bin  <- ifelse(act_fac == cls_name, 1, 0)
    r_obj    <- tryCatch(pROC::roc(act_bin, prob_mat[, i]), error = function(e) NULL)
    if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  })
  
  actual_counts <- as.numeric(table(act_fac))
  pred_counts   <- as.numeric(table(pred_fac))
  diff_vec      <- pred_counts - actual_counts
  diff_str      <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = levels(act_fac),
    Actual_Count = actual_counts,
    Pred_Count   = pred_counts,
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    ROC_AUC      = round(roc_auc_by_class, 4)
  )
  
  macro_prec    <- mean(prec_by_class)
  macro_rec     <- mean(rec_by_class)
  macro_roc_auc <- mean(roc_auc_by_class, na.rm = TRUE)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   HIERARCHICAL ENSEMBLE (XGB ESI1 + LGB ESI2345) - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy     : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision      : %.4f\n", macro_prec))
  cat(sprintf("  Macro Recall (Sens)  : %.4f\n", macro_rec))
  cat(sprintf("  Macro ROC-AUC        : %.4f\n", macro_roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Per-Class Performance & Count Comparison Summary:\n")
  print(report_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, macro_prec = macro_prec, macro_rec = macro_rec, macro_roc_auc = macro_roc_auc, report_df = report_df))
}
res_train <- evaluate_ensemble_5class(prob_train, train_df$target_layer1, "Train")
res_val   <- evaluate_ensemble_5class(prob_val,   val_df$target_layer1,   "Validation")
res_test  <- evaluate_ensemble_5class(prob_test,  test_df$target_layer1,  "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "meta_ensemble_esi1_esi2345_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "meta_ensemble_esi1_esi2345_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/meta_ensemble_esi1_esi2345_val_report.csv\n")
cat("Test CSV Report written to:       reports/meta_ensemble_esi1_esi2345_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Diagnostic Plots (Metrics Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Split           = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy        = c(res_train$acc,           res_val$acc,           res_test$acc),
  Macro_Precision = c(res_train$macro_prec,     res_val$macro_prec,     res_test$macro_prec),
  Macro_Recall    = c(res_train$macro_rec,      res_val$macro_rec,      res_test$macro_rec),
  Macro_ROC_AUC   = c(res_train$macro_roc_auc,  res_val$macro_roc_auc,  res_test$macro_roc_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Macro_Precision", "Macro_Recall", "Macro_ROC_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics (Hierarchical Meta-Ensemble)",
       subtitle = "Evaluating Overall Accuracy, Macro Precision, Macro Recall, and Macro ROC-AUC",
       y = "Metric Value Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "meta_ensemble_esi1_esi2345_metrics_barchart.png"), plot = p_bar, width = 9.5, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/meta_ensemble_esi1_esi2345_metrics_barchart.png\n")
p_bar